# 🚀 HỆ THỐNG AUTO-SCRIBE TRÊN COLAB (KẾT NỐI GOOGLE DRIVE)
Phiên bản này làm **TẤT CẢ TRÊN COLAB**, tự động truy cập vào Google Drive của bạn để bốc file SVG. 
Sau đó hiển thị Bảng điều khiển (Dashboard) để bạn **XEM TRƯỚC ẢNH VÀ ĐỔI ẢNH** trước khi xuất file.

### Hướng dẫn sử dụng:
1. Đảm bảo kho ảnh của bạn đã được tải lên Google Drive (VD: `My Drive/image/f`).
2. Upload file âm thanh `voiceover.mp3` vào Colab.
3. Chạy lần lượt các Cell bên dưới.



## BƯỚC 1: Cài Đặt Thư Viện & Kết Nối Google Drive


In [ ]:
from google.colab import drive
print("🔗 Đang yêu cầu quyền truy cập Google Drive...")
drive.mount('/content/drive')

print("⏳ Đang cài đặt thư viện lõi (Whisper, Gemini)...")
!apt-get install -y ffmpeg
!pip install -q openai-whisper google-genai requests

import whisper
import os
import json
import zipfile
import re
import shutil
import time
from google import genai
from google.genai import types
import IPython.display as display
import html
import random



## BƯỚC 2: Cấu Hình & Bóc Tách Thời Gian (Whisper)


In [ ]:
# --- CẤU HÌNH HỆ THỐNG ---
AUDIO_FILE = "voiceover.mp3"
GEMINI_API_KEY = "ĐIỀN_API_KEY_CỦA_BẠN_VÀO_ĐÂY"
ASSETS_DIR = "assets"
# Sửa lại đường dẫn này nếu thư mục ảnh trong Drive của bạn khác
DRIVE_IMAGE_DIR = "/content/drive/MyDrive/image/f" 

os.makedirs(ASSETS_DIR, exist_ok=True)

if not os.path.exists(AUDIO_FILE):
    print(f"\n❌ LỖI: Không tìm thấy file {AUDIO_FILE}! Vui lòng upload lên Colab (cột bên trái).")
else:
    print(f"\n⏳ Đang kiểm tra phần cứng và tải mô hình Whisper...")
    import torch
    device = "cuda" if torch.cuda.is_available() else "cpu"
    model_type = "small" if device == "cuda" else "base"
    print(f"🚀 Tối ưu hóa GPU: {'BẬT (T4/V100)' if device == 'cuda' else 'TẮT (Chỉ CPU)'} | Mô hình: Whisper {model_type.upper()}")
    model = whisper.load_model(model_type, device=device)

    print(f"🎙️ Đang bóc tách Audio: {AUDIO_FILE}...")
    result = model.transcribe(AUDIO_FILE, word_timestamps=True)

    MIN_SCENE_DURATION = 4.0
    scenes = []
    current_scene = {"start": 0, "end": 0, "text": ""}
    
    for segment in result['segments']:
        for word in segment['words']:
            if current_scene["text"] == "":
                current_scene["start"] = word["start"]
            current_scene["text"] += word["word"] + " "
            current_scene["end"] = word["end"]
            
            if (word["word"].strip().endswith(('.', '?', '!', ',')) and (current_scene["end"] - current_scene["start"]) >= MIN_SCENE_DURATION):
                scenes.append(current_scene)
                current_scene = {"start": 0, "end": 0, "text": ""}

    if current_scene["text"].strip() != "":
        scenes.append(current_scene)

    print(f"✅ Đã chia bài đọc thành {len(scenes)} phân cảnh (Scenes)!")



## BƯỚC 3: AI Lên Kịch Bản & Tự Động Rút Ảnh Từ Google Drive


In [ ]:
def search_local_svg(query, database_dir):
    stop_words = {"vector", "illustration", "clipart", "transparent", "icon", "svg", "drawing"}
    raw_words = re.sub(r'[^a-zA-Z0-9]', ' ', query).lower().split()
    query_words = set([w for w in raw_words if w not in stop_words and len(w) > 1])
    
    if not query_words: return None
    
    best_match = None
    max_score = 0
    
    for root, dirs, files in os.walk(database_dir):
        for f in files:
            if f.lower().endswith('.svg') or f.lower().endswith('.png'):
                clean_name = re.sub(r'[^a-zA-Z0-9]', ' ', os.path.splitext(f)[0]).lower()
                item_words = set(clean_name.split())
                
                score = len(query_words.intersection(item_words))
                if score > 0:
                    query_clean = " ".join(query_words)
                    if query_clean in clean_name:
                        score += 2.0
                        
                    if score > max_score:
                        max_score = score
                        best_match = os.path.join(root, f)
                        
    return best_match

def clean_json_response(text):
    text = text.strip()
    if text.startswith("```json"): text = text[7:]
    elif text.startswith("```"): text = text[3:]
    if text.endswith("```"): text = text[:-3]
    return text.strip()

client = genai.Client(api_key=GEMINI_API_KEY)
MODEL_ID = "gemini-2.5-flash" # Dùng 3.1-flash-lite nếu có
BATCH_SIZE = 5
scene_metadata = []

print("🤖 Đang phân tích kịch bản bằng Gemini và lục lọi ảnh trong Google Drive...")

if len(scenes) == 0:
    print("⚠️ Lỗi: Không có cảnh nào.")
else:
    for i in range(0, len(scenes), BATCH_SIZE):
        batch_scenes = scenes[i:i+BATCH_SIZE]
        prompt_scenes_text = "".join([f"ID: {i+j+1} | Text: \"{s['text']}\"\n" for j, s in enumerate(batch_scenes)])
            
        prompt_instruction = f"""
        Đạo diễn Whiteboard Animation. Câu thoại:
        {prompt_scenes_text}
        
        Trả về DUY NHẤT 1 MẢNG JSON ARRAY. Mỗi phần tử:
        - "id": ID câu thoại
        - "visual_concept": Ý tưởng (Tiếng Việt)
        - "svg_search_prompt": Từ khóa tìm kiếm hình ảnh NGẮN GỌN (1-3 từ Tiếng Anh)
        """
        
        for attempt in range(3):
            try:
                response = client.models.generate_content(
                    model=MODEL_ID, contents=prompt_instruction,
                    config=types.GenerateContentConfig(response_mime_type="application/json")
                )
                batch_data = json.loads(clean_json_response(response.text))
                
                for j, s in enumerate(batch_scenes):
                    scene_id = i + j + 1
                    result = next((item for item in batch_data if item.get("id") == scene_id), None)
                    if result:
                        meta = {
                            "start": s['start'], "end": s['end'], "speech_text": s['text'],
                            "visual_concept": result.get("visual_concept", ""),
                            "svg_search_prompt": result.get("svg_search_prompt", ""),
                            "file_name": f"scene_{scene_id:03d}.svg"
                        }
                        
                        # TÌM ẢNH TRONG GOOGLE DRIVE
                        best_match = search_local_svg(meta['svg_search_prompt'], DRIVE_IMAGE_DIR)
                        if best_match:
                            meta['file_name'] = f"scene_{scene_id:03d}{os.path.splitext(best_match)[1]}"
                            shutil.copy(best_match, os.path.join(ASSETS_DIR, meta['file_name']))
                            meta['source'] = f"✅ Đã bốc từ Drive: {os.path.basename(best_match)}"
                        else:
                            meta['source'] = "❌ KHÔNG TÌM THẤY ẢNH KHỚP (Cần tự chèn)"
                            # Tạo 1 file txt thế chỗ để đỡ lỗi
                            with open(os.path.join(ASSETS_DIR, f"scene_{scene_id:03d}_THIEU.txt"), 'w') as temp: temp.write("Thieu anh")
                            
                        scene_metadata.append(meta)
                break
            except Exception as e:
                print(f"Lỗi Batch {i//BATCH_SIZE+1}: {e}. Thử lại...")
                time.sleep(5)
        time.sleep(2)

    with open("scene_metadata.json", "w", encoding="utf-8") as f:
        json.dump(scene_metadata, f, ensure_ascii=False, indent=2)
    print("🎉 Hoàn tất bốc ảnh từ Drive!")



## BƯỚC 4: Bảng Điều Khiển Xem Trước (Visual Dashboard)


In [ ]:
# HIỂN THỊ TRỰC QUAN CÁC ẢNH AI ĐÃ CHỌN ĐỂ BẠN KIỂM DUYỆT
if not os.path.exists("scene_metadata.json"):
    print("Chưa có dữ liệu!")
else:
    with open("scene_metadata.json", "r", encoding="utf-8") as f:
        meta_data = json.load(f)

    html_code = '''
    <style>
      .dashboard { width: 100%; border-collapse: collapse; font-family: sans-serif; }
      .dashboard th, .dashboard td { border: 1px solid #ddd; padding: 12px; text-align: left; }
      .dashboard th { background-color: #f2f2f2; color: #333; }
      .preview-img { max-width: 150px; max-height: 150px; border: 1px solid #ccc; background: white; }
      .missing { color: red; font-weight: bold; }
    </style>
    <h3>🎬 BẢNG KIỂM DUYỆT ẢNH CHO VIDEOSCRIBE</h3>
    <p>💡 <i>Nếu bạn không thích ảnh nào do AI chọn, hãy mở thư mục <b>assets/</b> bên trái Colab, tải ảnh SVG/PNG của bạn lên, và đổi tên cho khớp với cột "Tên File Cần Đặt" để ghi đè.</i></p>
    <table class="dashboard">
      <tr>
        <th>Cảnh</th>
        <th>Nội dung Thoại</th>
        <th>Từ khóa AI tìm</th>
        <th>Trạng thái bốc ảnh (Drive)</th>
        <th>Tên File Cần Đặt</th>
        <th>Xem trước (Preview)</th>
      </tr>
    '''

    import urllib.parse
    for i, meta in enumerate(meta_data):
        file_path = os.path.join(ASSETS_DIR, meta['file_name'])
        
        # Nếu có ảnh thì hiển thị ảnh base64 lên bảng HTML
        img_html = "<span class='missing'>Không có ảnh</span>"
        if os.path.exists(file_path):
            try:
                import base64
                with open(file_path, "rb") as image_file:
                    encoded_string = base64.b64encode(image_file.read()).decode()
                ext = "svg+xml" if file_path.endswith(".svg") else "png"
                img_html = f"<img class='preview-img' src='data:image/{ext};base64,{encoded_string}' />"
            except:
                pass
                
        html_code += f'''
        <tr>
          <td><b>{i+1}</b></td>
          <td><i>"{meta['speech_text']}"</i></td>
          <td><b>{meta['svg_search_prompt']}</b></td>
          <td>{meta['source']}</td>
          <td><code>{meta['file_name']}</code></td>
          <td style="text-align:center;">{img_html}</td>
        </tr>
        '''
    
    html_code += "</table>"
    display.display(display.HTML(html_code))



## BƯỚC 5: Đóng Gói Ra File VideoScribe (.scribe)


In [ ]:
def generate_scribe_project():
    if not os.path.exists("scene_metadata.json"): return
    with open("scene_metadata.json", "r", encoding="utf-8") as f:
        meta_data = json.load(f)

    BUILD_DIR = "scribe_build"
    os.makedirs(BUILD_DIR, exist_ok=True)
    
    for item in os.listdir(BUILD_DIR):
        item_path = os.path.join(BUILD_DIR, item)
        if os.path.isfile(item_path): os.remove(item_path)
            
    if os.path.exists(AUDIO_FILE):
        shutil.copy(AUDIO_FILE, os.path.join(BUILD_DIR, "voiceover.mp3"))
        
    xml_path = os.path.join(BUILD_DIR, "drawing.xml")
    
    with open(xml_path, "w", encoding="utf-8") as f:
        f.write('<?xml version="1.0" encoding="UTF-8"?>\n<drawing>\n')
        
        for i, meta in enumerate(meta_data):
            file_path = os.path.join(ASSETS_DIR, meta['file_name'])
            # Hỗ trợ ghi đè file có đuôi khác
            alt_svg = os.path.join(ASSETS_DIR, meta['file_name'].split('.')[0] + '.svg')
            alt_png = os.path.join(ASSETS_DIR, meta['file_name'].split('.')[0] + '.png')
            
            if os.path.exists(alt_svg): file_path = alt_svg
            elif os.path.exists(alt_png): file_path = alt_png
            elif not os.path.exists(file_path): continue
            
            is_svg = file_path.endswith('.svg')
            actual_filename = os.path.basename(file_path)
            
            if is_svg:
                with open(file_path, "r", encoding="utf-8") as f2:
                    content = re.sub(r'<\?xml[^>]*\?>', '', f2.read())
            else:
                shutil.copy(file_path, os.path.join(BUILD_DIR, actual_filename))
                content = f"<image src='{actual_filename}' width='800' height='600' />"
                
            duration = meta['end'] - meta['start']
            
            pos_x = 1200 * (i % 3)
            pos_y = 800 * int(i / 3)
            
            elem_type = 'drawing' if is_svg else 'image'
            
            # Dynamic Animation Engine v2.0
            import random
            if is_svg:
                effect_choice = random.choices(['normal', 'movein'], weights=[70, 30])[0]
            else:
                effect_choice = random.choices(['movein', 'fadein'], weights=[80, 20])[0]
            
            draw_style = f'draw_style_{effect_choice}'
            movin_compass = str(random.randint(1, 8))
            movin_flow = random.choices(['0', '2'], weights=[70, 30])[0] # 0: smooth, 2: bounce
            movin_arc = random.choices(['1', '2'], weights=[70, 30])[0] # 1: straight, 2: curved
            draw_detail = random.choice(['yes', 'no'])
            hand_choice = random.choice(['', 'default_nohand']) # '' means default hand (tay đưa vào), 'default_nohand' (tự bay vào)
            
            if effect_choice == 'movein' or effect_choice == 'fadein':
                target_time_ms = random.randint(500, 1500) # Fast pop-in
            else:
                target_time_ms = max(1000, int((duration - 1.0) * 1000)) # Slow draw
                
            # Random Filter (10% chance for greyscale)
            filter_str = "<filters/>"
            if random.random() < 0.10:
                filter_str = '''<filters>\n  <filter filterType="greyscale" distance="4" amount="30" blurX="4" blurY="4" angle="120" colour="0"/>\n</filters>'''
            
            f.write(f'''  <element elementType="{elem_type}" drawStyle="{draw_style}" customHandMD5="{hand_choice}" movinCompass="{movin_compass}" movinFlow="{movin_flow}" movinArc="{movin_arc}" movinAllowRotate="yes" drawDetail="{draw_detail}" targetTime="{target_time_ms}" pauseTime="500" transitionTime="500" currentPosX="{pos_x}" currentPosY="{pos_y}" cameraPositionX="{pos_x}" cameraPositionY="{pos_y}" cameraScale="1.0">\n''')
            f.write(f'''    {filter_str}\n''')
            if is_svg:
                f.write(f'''    <drawingXML><![CDATA[{content}]]></drawingXML>\n''')
            else:
                f.write(f'''    <imageRef>{actual_filename}</imageRef>\n''')
            f.write('''  </element>\n''')
            
        if os.path.exists(AUDIO_FILE):
            f.write('''  <audio volume="1.0" loop="false">\n    <file>voiceover.mp3</file>\n  </audio>\n''')
            
        f.write('</drawing>')

    output_filename = "Auto_Project.scribe"
    with zipfile.ZipFile(output_filename, 'w', zipfile.ZIP_DEFLATED) as zipf:
        for root, dirs, files in os.walk(BUILD_DIR):
            for file in files:
                file_path = os.path.join(root, file)
                arcname = os.path.relpath(file_path, BUILD_DIR)
                zipf.write(file_path, arcname)
                    
    print(f"\n🎉 ĐÃ HOÀN TẤT! File dự án của bạn: {output_filename}")

generate_scribe_project()

